In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten/o4_mini_high_small_q5/checkpoints/post_cell_42_try_0.pickle

In [ ]:
%%cudf.pandas.profile
### cell 43 ###

def combine_subset_of_data_from_multiple_years_55(question_of_interest, x_axis_title, include_2017=False):
    # Map each year to its responses dataframe
    mapping = {
        '2017': responses_df_2017,
        '2018': responses_df_2018_cell10,
        '2019': responses_df_2019_cell10,
        '2020': responses_df_2020,
        '2021': responses_df_2021,
        '2022': responses_df_2022_cell10
    }
    # Select years in the correct order
    years = ['2017','2018','2019','2020','2021','2022'] if include_2017 else ['2018','2019','2020','2021','2022']
    dfs = []
    for year in years:
        df_year = mapping[year]
        # Compute percentages entirely on GPU
        s = df_year[question_of_interest].value_counts(dropna=False)
        s = (s * 100 / df_year[question_of_interest].count()).round(1).sort_index()
        # Convert Series to DataFrame and add columns
        df_tmp = s.reset_index()
        df_tmp.columns = [x_axis_title, 'percentage']
        df_tmp['year'] = year
        dfs.append(df_tmp[['percentage', 'year', x_axis_title]])
    # Concatenate all years in one GPU-backed operation
    df_combined = pd.concat(dfs, ignore_index=True)
    return df_combined

question_of_interest_cell55 = (
    'Select the title most similar to your current role (or most recent title if retired): - Selected Choice'
)
job_df_combined = combine_subset_of_data_from_multiple_years_55(
    question_of_interest_cell55,
    title_for_x_axis_cell43,
    include_2017=False
)
# Standardize the job title and filter on GPU
job_df_combined[title_for_x_axis_cell43] = (
    job_df_combined[title_for_x_axis_cell43]
    .replace('Data Analyst (Business, Marketing, Financial, Quantitative, etc)', 'Data Analyst')
)
job_df_combined = job_df_combined[job_df_combined[title_for_x_axis_cell43].isin([
    'Data Analyst', 'Data Engineer', 'Data Scientist',
    'Research Scientist', 'Software Engineer'
])]
job_df_combined.info()